In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from typing import List, Union, Tuple, Dict, Optional, Literal, Any
import re
from scipy.optimize import minimize
from IPython.display import display, clear_output
import ipywidgets as widgets
from tqdm.notebook import tqdm
import seaborn as sns

%matplotlib widget

In [2]:
class Spectra():
    def __init__(self,
                 path_to_spectra:Union[str,Path],
                 delimiter=',',
                 skiprows:int=1):
        
        self.path_to_spectra = Path(path_to_spectra)
        self.x, self.y = Spectra.load_spectra(path_to_spectra=self.path_to_spectra,
                                              delimiter=delimiter,
                                              skiprows=skiprows)

    
    @staticmethod
    def load_spectra(path_to_spectra:Union[str,Path], delimiter=',', skiprows:int=1):
        '''
        Функция возвращает массивы аргументов и значений спектра
        '''
        spectra = np.loadtxt(path_to_spectra, delimiter=delimiter, skiprows=skiprows)
        return spectra[:,0],spectra[:,1]
    
    
    def plot_spectra(self):
        plt.figure(figsize=(7,5))
        plt.plot(self.x, self.y, 'b-',label='Spectra')
        plt.legend()
        plt.show()
    
    def crop_spectra(self, x_start:float, x_end:float):
        '''
        Функция обрезает спектр таким образом, что в рассмотрении остаются только точки
        лежащие в интервале значений аргумента функции от x_start до x_end включительно
        '''
        new_x = self.x[np.logical_and(self.x>=x_start,self.x<=x_end)]
        new_y = self.y[np.logical_and(self.x>=x_start,self.x<=x_end)]
        self.x, self.y = new_x, new_y
    
    @staticmethod
    def get_value_by_indexes(x:np.ndarray, indecies: np.ndarray):
        '''
        Возвращает значения массива x для указанного массива индексов indexes
        '''
        return x[indecies]
    
    @staticmethod
    def get_indexes_by_values(x:np.array, values:np.array):
        '''
        Возвращает индекс наиблежайшего элемента массива к значениям values
        '''
        return np.argmin(abs(x[:,None]-values), axis=0)
    
    def set_maxima(self, tolerance:int=3, max_from_U:np.ndarray=None):
        '''
        Функция позволяет в ручную на графике задать положения максимумов поглощения в спектре
        tolerance - определяет диапазон индексов для поиска максимума относительно выделенной пользователем точки
        max_from_U - не обязательый параметр, хранящий положения максимумов поглощения из Data_U
        '''
        maxima_values = []
        maxima_indecies = []

        fig, ax = plt.subplots(constrained_layout=True, figsize=(7, 5))
        fig.canvas.toolbar_position = 'bottom'
        
        def replot():
            ax.clear()
            ax.plot(self.x, self.y,'b-', label='Spectra')
            ax.vlines(maxima_values, ymin=min(self.y), ymax=max(self.y), color='red', label='Maxima')
            if max_from_U is not None:
                ax.vlines(max_from_U, ymin=min(self.y), ymax=max(self.y), color='green', label='From U', alpha=0.7)
                ax.set_xlim(self.x.min(), self.x.max())
            ax.legend()

        def accept(change):
            fig.canvas.mpl_disconnect(cid)
            self.maxima_values = np.array(maxima_values, dtype=np.float32)
            self.maxima_indecies = np.array(maxima_indecies, dtype=np.int32)
            clear_output()

        def del_last(change):
            maxima_values.pop()
            del_btn.layout.visibility = 'visible' if len(maxima_values)>0 else 'hidden'
            replot()
            
        def find_nearest_maxima(value:float):
            nearest_value_idx = Spectra.get_indexes_by_values(x=self.x, values=value)
            start_index = int(max(0,nearest_value_idx-tolerance))
            end_index = int(min(nearest_value_idx+tolerance, self.x.shape[0]-1))
            x_max = self.x[self.y==max(self.y[start_index:end_index])]
            x_max = list(set(self.x[start_index:end_index]) & set(x_max))[0]
            idx_max = Spectra.get_indexes_by_values(x=self.x, values=value)[0]
            return x_max, idx_max


        def graph_onclick(event):
            nonlocal maxima_values
            nonlocal del_btn
            if event.inaxes==ax:
                ix = event.xdata
                val, idx = find_nearest_maxima(ix)
#                 maxima_values.append(ix)
                maxima_values.append(val)
                maxima_indecies.append(idx)
                replot()
                del_btn.layout.visibility  = 'visible'

        accept_btn = widgets.Button(description = 'Accept')
        accept_btn.on_click(accept)

        del_btn = widgets.Button(description = 'Del last')
        del_btn.on_click(del_last)
        del_btn.layout.visibility  = 'hidden'

        ui = widgets.HBox([accept_btn,del_btn])
        display(ui)

        cid = fig.canvas.mpl_connect('button_press_event', graph_onclick)
        replot()
        return maxima_values,maxima_indecies

In [3]:
class CarnalDataPrepare():
    def __init__(self, init_U_data:pd.DataFrame):
        '''
        Функция принимает на вход DataFrame со следующими данными:
        TERM1	TERM2	(U2)*2	(U4)*2	(U6)*2	LEVEL1	LEVEL2.
        В результате инициализации класса будут сгенерированы следующие два аттрибута:
        self.lower - pd.DataFrame содержащий переходы из I-го уровня во все ниже лежащие уровни;
        self.ground - pd.DataFrame содержащий переходы в основное состояние.
        '''
        self.init_data = init_U_data
        self.prepared_data = self.prepare_df()
        self.lower = self.prepare_lower()
        self.ground = self.prepare_ground()

    def prepare_df(self):
        df = self.init_data.copy()
        def term_concat(df):
            return df.TERM1+'-'+df.TERM2
        def get_J(term:str)->np.ndarray:
            '''
            Ожидается что темр записан корректно в виде $^{2S+1}L_J$
            Если указывается переход то в виде Term_initial-Term_final, но будет возвращена информация для терма final состояния.
            Функция возвращает np.array([S,L,J])
            '''
            L_dict = {'S':0, 'P':1, 'D':2, 'F':3, 'G':4, 'H':5, 'I':6, 'K':7, 'L':8, 'M':9, 'N':10, 'O':11, 'Q':12, 'R':13, 'T':14, 'U':15, 'V':16,}
            final_term =  re.split('-',term)[1] if len(re.split('-',term))>1 else term
            assert re.fullmatch('\d*[A-Z]{1}\d*/\d*', final_term) is not None, f'Wrong Term - {final_term}'
            l_key = re.search('[A-Z]{1}',final_term)[0]
            assert l_key in L_dict, f'Incorrect L value - {l_key}'
            _, j_str = re.split(l_key, final_term)
            j_numerator,j_denominator = re.split('/', j_str)
            J = float(j_numerator)/float(j_denominator)
            return J
        def cm2eV2nm(gap:float):
            try:
                return 1e7/gap
            except:
                return None
        
        df['Energy']=df['LEVEL1']-df['LEVEL2']
        df['J_i'] = df.TERM1.apply(lambda x: get_J(x))
        df['J_f'] = df.TERM2.apply(lambda x: get_J(x))
        df['Term'] = df.apply(lambda x:term_concat(x),axis=1)
        df['Lambda'] = df.Energy.apply(lambda x: cm2eV2nm(x))
        df.rename(columns={'(U2)*2':'U2','(U4)*2':'U4','(U6)*2':'U6', 'TERM1':'TERM_i','TERM2':'TERM_f'},inplace=True)
        return df

    def prepare_ground(self):
        df = self.prepared_data.copy()
        ground = df.LEVEL2.min()
        ground_df = df.query(f'LEVEL2=={ground} and LEVEL1!={ground}')[['TERM_i','TERM_f','Term','Energy','U2','U4','U6','Lambda','J_i','J_f']]
        ground_df = ground_df.sort_values(by=['Lambda'], ascending=True)
        ground_df = ground_df.reset_index(drop=True)
        return ground_df

    def prepare_lower(self):
        df = self.prepared_data.copy()
        lower_df = df.query(f'Energy>0')[['TERM_i','TERM_f','Term','Energy','U2','U4','U6','Lambda','J_i','J_f']]
        lower_df = lower_df.reset_index(drop=True)
        return lower_df

In [4]:
class Data_U():
    def __init__(self,
                 path_to_U_data:Union[str,Path]):
        
        self.path_to_U_data = Path(path_to_U_data)
        self.data_U = self._load_U_data()

    @staticmethod
    def term_parser(term:str)->np.ndarray:
        '''
        Ожидается что темр записан корректно в виде $^{2S+1}L_J$
        Если указывается переход то в виде Term_initial-Term_final, но будет возвращена информация для терма final состояния.
        Функция возвращает np.array([S,L,J])
        '''

        L_dict = {'S':0, 'P':1, 'D':2, 'F':3, 'G':4, 'H':5, 'I':6, 'K':7, 'L':8, 'M':9, 'N':10, 'O':11, 'Q':12, 'R':13, 'T':14, 'U':15, 'V':16,}
        final_term =  re.split('-',term)[1] if len(re.split('-',term))>1 else term
        assert re.fullmatch('\d*[A-Z]{1}\d*/\d*', final_term) is not None, f'Wrong Term - {final_term}'
        l_key = re.search('[A-Z]{1}',final_term)[0]
        assert l_key in L_dict, f'Incorrect L value - {l_key}'
        L = L_dict[l_key]
        s2_1, j_str = re.split(l_key, final_term)
        S = (float(s2_1)-1)/2
        j_numerator,j_denominator = re.split('/', j_str)
        J = float(j_numerator)/float(j_denominator)
        return np.array([S,L,J])
    
    def _load_U_data(self):
        '''
        Загружает файл с значениями приведенной матрицы U для анализа Judd-Ofelt'а.
        Обязательные столбцы: Term, U2, U4, U6.
        Если есть столбец Lambda содержащий значения длин волн (nm) соответствующих максимумам поглащения
        то столбец Energy (cm-1) не обязателен, в противном случае он должен присутствовать.
        Функция возвращает pandas DataFrame с следующими данными:
            Term, U2, U4, U6, Energy (?), Lambda, S_f, L_f, J_f
        где S_f, L_f, J_f - квантовые числа соответствующие финальному состоянию
        '''
        u_data = pd.read_csv(self.path_to_U_data,).reset_index(drop=True)#.drop(columns=['Unnamed: 0'])
        assert 'Energy' in u_data.keys() or 'Lambda' in u_data.keys(), 'Отсуствуют данные об энергиях или длиннах волн для переходов'
        assert {'Term','U2','U4','U6'}<=set(u_data.keys()), 'Столбцы Term, U2, U4, U6 являются обязательными'
        assert min(u_data.isnull().sum()) == 0, 'В наборе данных присутствуют пропущенные данные'

        if 'Lambda' not in u_data.keys():
            u_data['Lambda'] = u_data.Energy.apply(lambda x:1e7/x)

        s_l_j = np.array([Data_U.term_parser(x) for x in u_data.Term])
        u_data[['S_f','L_f','J_f']] = s_l_j
        
        u_data = u_data.sort_values(by=['Lambda'], ascending=True)
        return u_data

In [5]:
class Gauss():
    @staticmethod
    def slice_spectra_by_boundary(x:np.ndarray, y:np.ndarray, boundary_indexes:np.ndarray)->Tuple[List[np.ndarray]]:
        '''
        Функция принимает значения аргумента функции x, значения функции y, и индексы границ для нарезки спектра на сегменты,
        возвращает списки new_x и new_y, которые содержат массивы нарезанных значений x и y
        '''
        assert x.shape==y.shape,'x and y must have the same shape'
        new_x = [x[start:end] for start, end in boundary_indexes]
        new_y = [y[start:end] for start, end in boundary_indexes]

        return new_x, new_y
    
    @staticmethod
    def get_gauss_parameters(x:np.ndarray, y:np.ndarray, boundary_indexes:np.ndarray)->np.ndarray:
        '''
        Функция принимает значения аргумента функции x, значения функции y, и индексы границ для нарезки спектра на сегменты,
        возвращает массив стартовых параметров для функции Гаусса для каждого сегмента
        '''
        parameters = np.array([[y[start:end].max(), x[start:end].mean(), (x[end]-x[start])/6] for start,end in boundary_indexes])
        return parameters
    
    @staticmethod
    def gauss(amplitude:float, mean:float, std:float, x:np.ndarray)->np.ndarray:
        '''
        Функция возвращает значения функции Гаусса определяемой параметрами amplitude, mean, std для значений аргумента x
        '''
        return amplitude*np.exp((-1*(x-mean)**2)/(2*(std**2)))
    
    @staticmethod
    def gauss_integral(amplitude:float, mean:float, std:float)->float:
        '''
        Функция возвращает аналитический результат интегрирования одномерной функции Гаусса определяемой параметрами amplitude, mean и std
        '''
        return np.sqrt(2*np.pi)*amplitude*abs(std)
    
    @staticmethod
    def gauss_fit(func_val:np.ndarray,
                  func_arg:np.ndarray,
                  num_gauss:int=1,
                  gauss_parameters:np.ndarray=None)->dict:
        '''
        Функция принимает массивы аргументов и значений функции - func_arg, func_val
        количество Гауссовских пиков для фиттинга - num_gauss
        и стартовые параметры для фиттинга - gauss_parameters
        Если gauss_parameters не указаны, то параметры считаются равными 1

        Функция возвращает словарь содержащий:
            параметры функций Гаусса после фиттинга - gauss_parameters
            параметры линейной функции фона (y1,y2) - Linear_background_parameters
            (y2-y1)*((func_arg-func_arg[0])/(func_arg[-1]-func_arg[0]))+y1
            и значение результирующей функции после фиттинга (bkg+Gauss) - Result_function_values
        '''

        assert func_val.size==func_arg.size, 'вектора значений и аргументов функции должны иметь одинаковый размер'

        parameters = np.append(gauss_parameters, (1,1))


        gauss_bounds = [(1e-3,2*max(func_val)),(func_arg[0],func_arg[-1]),(1e-3,func_arg[-1]-func_arg[0])]*num_gauss
        bkg_bounds = [(0,None),(0,None)]
        bounds = np.array(gauss_bounds+bkg_bounds)

        func = lambda params : np.array([Gauss.gauss(amplitude, mean, std, func_arg) for amplitude, mean, std in params]).sum(axis=0)
        bkg_func = lambda y1,y2 : (y2-y1)*((func_arg-func_arg[0])/(func_arg[-1]-func_arg[0]))+y1

        def error(parameters):
            y1,y2 = parameters[-2],parameters[-1]
            gauss_parameters = parameters[:-2]
            gauss_parameters = gauss_parameters.reshape((num_gauss,3))

            res_func = bkg_func(y1,y2) + func(gauss_parameters)


            return ((func_val-res_func)**2).sum()

        res = minimize(error, parameters, method='L-BFGS-B', bounds=bounds, options={'gtol': 1e-20, 'disp': True})

        result_dic = dict()

        y1,y2 = res.x[-2],res.x[-1]
        gauss_parameters = res.x[:-2]

        gauss_parameters = gauss_parameters.reshape((num_gauss,3))

        result_func_val = bkg_func(y1,y2) + func(gauss_parameters)

        result_dic['Gaussian_parameters'] = gauss_parameters
        result_dic['Linear_background_parameters'] = (y1,y2)
        result_dic['Result_function_values'] = result_func_val

        return result_dic

In [6]:
def make_box_layout():
     return widgets.Layout(
        border='solid 5px black',
        margin='0px 10px 10px 0px',
        padding='5px 5px 5px 5px'
     )


class Plot_Segment():
    def __init__(self,
                 spectra_args:np.ndarray,
                 spectra_values:np.ndarray,
                 max_wls:float,
                 boundaries:np.ndarray,
                 grouped:bool,
                 mean_from_U:float,
                 extension:float=20.,
                 text:str='',
                 ):
        super().__init__()
    
        self.spectra_args = spectra_args
        self.spectra_values = spectra_values
        self.text = text
        self.max_wls = max_wls
        self.maxima_values = []
        self.mean_from_U = mean_from_U
        self.grouped = grouped
        
        self.fig, self.ax = plt.subplots(constrained_layout=True, figsize=(5, 3.5))

        self.fig.canvas.toolbar_position = 'bottom'
       
        bound_start_idx, bound_end_idx = boundaries
        
        self.results = {'Boundaries_start_idx':bound_start_idx,
                        'Boundaries_end_idx':bound_end_idx,
                        'Num_Gauss': 1}
        
        self.accepted = False
        
        plot_start_idx = np.argmin(abs(self.spectra_args-(self.spectra_args[bound_start_idx]-extension)))
        plot_end_idx = np.argmin(abs(self.spectra_args-(self.spectra_args[bound_end_idx]+extension)))
    
        self.replot(plot_boundaries=(plot_start_idx,plot_end_idx), analysis_boundaries=(bound_start_idx,bound_end_idx))
        
        plot_boundaries_range_slider = widgets.IntRangeSlider(value=(plot_start_idx, plot_end_idx), 
                                                        min=0, max=self.spectra_args.shape[0]-1, step=1, 
                                                        description='Plt idxs'
                                                       )
        
        analysis_boundaries_range_slider = widgets.IntRangeSlider(value=(bound_start_idx, bound_end_idx), 
                                                        min=0, max=self.spectra_args.shape[0]-1, step=1, 
                                                        description='Bnds idxs'
                                                       )
        
        num_gauss_dropdown = widgets.Dropdown(value=1, 
                                              options=[0, 1, 2, 3, 4, 5, 6, 7], 
                                              description='Num Gauss'
                                             )
        
        self.ngd = num_gauss_dropdown
        
        accept_button = widgets.Button(description='Accept')
        accept_button.on_click(self.accept)
        accept_button.layout.visibility = 'hidden'
        self.accept_button = accept_button
        
        
        
        paired_checkbox = widgets.Checkbox(description='Paired',
                                           value=False)
        
        grouped_checkbox = widgets.Checkbox(description='Grouped',
                                           value=bool(self.grouped))
        
        ##########################################################################
        fit_button = widgets.Button(description='Fit segment')
        fit_button.on_click(self.fit)
        self.fit_button = fit_button
        
        refit_button = widgets.Button(description='Refit')
        refit_button.on_click(self.refit)
        refit_button.layout.visibility = 'hidden'
        self.refit_button = refit_button
        
        set_maxima_button = widgets.Button(description='Set maxima')
        set_maxima_button.on_click(self.set_maxima)
        self.set_maxima_button = set_maxima_button
        
        del_last_maxima_button = widgets.Button(description='Del last maxima')
        del_last_maxima_button.on_click(self.del_last)
        del_last_maxima_button.layout.visibility = 'hidden'
        self.del_last_maxima_button = del_last_maxima_button
        ###########################################################################
        
        boundaries_ui = widgets.VBox([plot_boundaries_range_slider,          #0
                                      analysis_boundaries_range_slider,      #1
                                      num_gauss_dropdown,                    #2
                                      paired_checkbox,
                                      grouped_checkbox,])                     #4
        
#         boundaries_ui.layout = make_box_layout()
        
        fit_ui = widgets.VBox([set_maxima_button,                            #0
                               del_last_maxima_button,                       #1
                               fit_button,                                   #2
                               refit_button,])                               #3

#         fit_ui.layout = make_box_layout()
        
        ##############################################################################
        num_paired = widgets.Dropdown(value=1, 
                                      options=list(range(1,self.ngd.value+1)), 
                                      description='Num Paired Gauss'
                                     )
        num_paired.layout = make_box_layout()
        num_paired.layout.visibility = 'hidden'
        self.n_p_g = num_paired
        ##############################################################################
        
        control_ui = widgets.HBox([boundaries_ui,
                                   fit_ui,
                                   num_paired])
        
#         control_ui.layout = make_box_layout()
        

        self.ui = widgets.VBox([control_ui,
                                accept_button])
        
#         self.ui.layout = make_box_layout()

        
        self.out = widgets.interactive_output(self.replot, {'plot_boundaries':plot_boundaries_range_slider, 
                                                            'analysis_boundaries':analysis_boundaries_range_slider,
                                                           })

        display(self.out, self.ui)
         
    
    def replot(self, plot_boundaries, analysis_boundaries):
        
        plot_start_idx, plot_end_idx = plot_boundaries
        bound_start_idx, bound_end_idx = analysis_boundaries
        
        self.ax.clear()
        self.ax.set_title(self.text)
        self.spectra, = self.ax.plot(self.spectra_args, self.spectra_values, 'b-', label='Spectra')

        self.ax.set_xlim(self.spectra_args[plot_start_idx], self.spectra_args[plot_end_idx])
        
        y_min_on_segment = self.spectra_values[plot_start_idx:plot_end_idx].min()
        y_max_on_segment = self.spectra_values[plot_start_idx:plot_end_idx].max()
        self.ax.set_ylim(y_min_on_segment, y_max_on_segment*1.1)
        
        self.ax.vlines(self.maxima_values, ymin=y_min_on_segment, ymax=y_max_on_segment, color='purple', label='Maxima', alpha=0.7)
        self.ax.vlines([self.mean_from_U], ymin=y_min_on_segment, ymax=y_max_on_segment, color='black', ls=':', label='Mean U', alpha=0.7)
        
        x_start, x_end = self.spectra_args[bound_start_idx],self.spectra_args[bound_end_idx]
        self.max_line, = self.ax.plot([self.max_wls, self.max_wls],[y_min_on_segment, y_max_on_segment],color='black',ls='-',alpha=0.2, label='max')
        self.start_boundary_line, = self.ax.plot([x_start, x_start],[y_min_on_segment, y_max_on_segment],color='green',ls='-',alpha=0.2, label='start')
        self.end_boundary_line, = self.ax.plot([x_end, x_end],[y_min_on_segment, y_max_on_segment],color='red',ls='-',alpha=0.2, label='end')
        self.ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

    def plot_fit_res(self):
        plot_start_idx, plot_end_idx = self.ui.children[0].children[0].children[0].value
        bound_start_idx, bound_end_idx = self.ui.children[0].children[0].children[1].value
        x = self.spectra_args[bound_start_idx:bound_end_idx]
        y = self.spectra_values[bound_start_idx:bound_end_idx]
        
        y_min_on_segment = self.spectra_values[plot_start_idx:plot_end_idx].min()
        y_max_on_segment = self.spectra_values[plot_start_idx:plot_end_idx].max()
        
        self.ax.clear()
        self.ax.set_title(self.text)
        self.spectra, = self.ax.plot(self.spectra_args, self.spectra_values, 'b-', label='Spectra')
        
        self.ax.set_xlim(self.spectra_args[plot_start_idx], self.spectra_args[plot_end_idx])
        self.ax.set_ylim(0, y_max_on_segment*1.4)
        
        
        guass_x = np.arange(self.spectra_args[plot_start_idx], self.spectra_args[plot_end_idx],0.1)
        y_fitted = self.fit_res['Result_function_values']
        
        y1, y2 = self.results['Bkg_parameters']

        bkg = (y2-y1)*((guass_x-x[0])/(x[-1]-x[0]))+y1

        for amplitude, mean, std in self.results['Gauss_parameters']:
            gauss_y = np.array(Gauss.gauss(amplitude, mean, std, guass_x))
            self.ax.plot(guass_x, gauss_y, color='black', ls=':' ,label='Gauss')
            self.ax.fill_between(guass_x, gauss_y+bkg, bkg, color='red', alpha=0.3)

        self.ax.plot(x,y_fitted,'r:',label='Fitted')
        self.ax.plot(guass_x, bkg, color='purple', ls='--' ,label='Bkg')
        self.ax.plot(x,y-y_fitted,'g-.',label='Subtracted')
        
        self.ax.vlines([self.results['Lambda_mean']], ymin=y_min_on_segment, ymax=y_max_on_segment, color='black', label='lambda mean')
        self.ax.vlines([self.mean_from_U], ymin=y_min_on_segment, ymax=y_max_on_segment, color='green', ls='-', label='Mean U', alpha=0.7)
        self.ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        
    def graph_onclick(self, event):
        if event.inaxes==self.ax:
            ix = event.xdata
            self.maxima_values.append(ix)
            self.replot(self.ui.children[0].children[0].children[0].value,self.ui.children[0].children[0].children[1].value)
            self.del_last_maxima_button.layout.visibility  = 'visible'
            
            n_click = self.ui.children[0].children[0].children[2].value
            if len(self.maxima_values)==n_click:
                self.fig.canvas.mpl_disconnect(self.cid)
    
    
    def set_maxima(self, change):
        self.cid = self.fig.canvas.mpl_connect('button_press_event', self.graph_onclick)
        change.layout.visibility='hidden'
        
        paired = self.ui.children[0].children[0].children[3].value
        if paired:
            self.n_p_g.options=list(range(1,self.ngd.value+1))
            self.n_p_g.layout.visibility = 'visible'

   
    def del_last(self, change):
        self.maxima_values.pop()
        change.layout.visibility = 'visible' if len(self.maxima_values)>0 else 'hidden'
        self.cid = self.fig.canvas.mpl_connect('button_press_event', self.graph_onclick)
        self.replot(self.ui.children[0].children[0].children[0].value,self.ui.children[0].children[0].children[1].value)

    def fit(self, change):
        
        change.layout.visibility='hidden'
        self.del_last_maxima_button.layout.visibility='hidden'
        start_idx = self.ui.children[0].children[0].children[1].value[0]
        end_idx = self.ui.children[0].children[0].children[1].value[1]
        n_gauss = self.ui.children[0].children[0].children[2].value
        
        if n_gauss==0:
            self.results = {'Boundaries_start_idx': 0,
                            'Boundaries_end_idx'  : 0,
                            'Num_Gauss'           : 0,
                            'Paired'              : False,
                            'Grouped'             : False,
                            'Gauss_parameters'    : [[0,0,0]],
                            'Lambda_mean'         : 0,
                            'Gauss_AUC'           : 0,
                            'Bkg_parameters'      : (0,0),
                           }
            self.accept_button.layout.visibility  = 'visible'
        else:
            temp_x = self.spectra_args[start_idx:end_idx]
            temp_y = self.spectra_values[start_idx:end_idx]
            std = (temp_x.max()-temp_x.min())/n_gauss/6
            means = self.maxima_values
            amplitudes =[temp_y[np.argmin(abs(temp_x-x))] for x in means]
            gauss_params = np.array([[amplitude, mean, std] for amplitude,mean in zip(amplitudes,means)])

            fit_res = Gauss.gauss_fit(func_val=temp_y,
                                           func_arg=temp_x,
                                           num_gauss=n_gauss,
                                           gauss_parameters=gauss_params)
            self.fit_res = fit_res

            paired = self.ui.children[0].children[0].children[3].value
            grouped = self.ui.children[0].children[0].children[4].value

            self.results = {'Boundaries_start_idx': start_idx,
                            'Boundaries_end_idx'  : end_idx,
                            'Num_Gauss'           : n_gauss,
                            'Paired'              : paired,
                            'Grouped'             : grouped,
                           }

            if paired:
                gauss_params = np.round(fit_res['Gaussian_parameters'],10)
                gauss_params = gauss_params[:self.n_p_g.value]
                self.results['Gauss_parameters']=gauss_params
            else:
                gauss_params = np.round(fit_res['Gaussian_parameters'],10)
                self.results['Gauss_parameters']=gauss_params
                
            
            gauss_auc = np.array([Gauss.gauss_integral(amplitude, mean, std) for amplitude, mean, std in gauss_params]).sum()
            self.results['Gauss_AUC']=gauss_auc
            
            if grouped:
                self.results['Lambda_mean'] = self.mean_from_U
            else:
                lambda_mean = np.array([Gauss.gauss_integral(amplitude, mean, std)*mean for amplitude, mean, std in gauss_params]).sum()/gauss_auc
                self.results['Lambda_mean']=lambda_mean


            y1, y2 = fit_res['Linear_background_parameters']
            self.results['Bkg_parameters']= (y1, y2)

            self.plot_fit_res()
            self.accept_button.layout.visibility  = 'visible'
            self.refit_button.layout.visibility = 'visible'

        
    def refit(self, change):
        self.results = dict()
        self.maxima_values = []
        self.replot(self.ui.children[0].children[0].children[0].value,self.ui.children[0].children[0].children[1].value)
        self.accept_button.layout.visibility  = 'hidden'
        self.set_maxima_button.layout.visibility = 'visible'
        self.fit_button.layout.visibility = 'visible'
        self.n_p_g.layout.visibility = 'hidden'
        change.layout.visibility = 'hidden'

 
    def accept(self,change):
        self.out.clear_output()
        self.ui.close()
        clear_output(wait=True)
        self.accepted = True

In [7]:
class Judd_Ofelt():
    def __init__(self,
                 U_data:Data_U,
                 spectra:Spectra,
                 ):
        
        self.U_data=U_data.data_U
        self.spectra=spectra
        
    
    def define_shift(self, tol:float=30.)->float:
        '''
        Функция определяет сдвиг между положениями максимумов из U_data и spectra (длины волн максимумов)
        т.е. возвращает значение на которое нужно сместить max_from_U, для наилучшего соответствия с max_from_spectra
        tol - параметр чувствительности, если на заданном расстоянии от максимума из max_from_U
        отсутствуют максимумы из max_from_spectra, то данный пик убирается из рассмотрения
        '''
        max_from_U = self.U_data.Lambda.values
        try:
            max_from_spectra = self.spectra.maxima_values
        except AttributeError:
            print('Error: attribute "maxima_values" not found, use spectra.set_maxima')
            return None

        params = 0
        def error(params):
            new_U = max_from_U+params
            init_dif = abs(new_U[:,None] - max_from_spectra)
            init_dif = np.array([init_dif[i] for i in range(init_dif.shape[0]) if init_dif[i].min()<tol])
            min_dif =np.min(init_dif, axis=1)
            error = min_dif.sum()
            return error

        res = minimize(error, params, method='BFGS', options={'gtol': 1e-20, 'disp': True})
        self.shift = res.x[0]
        return res.x[0]
    
    def get_correct_maxima_df(self,
                              del_tol:float=30,
                              merge_tol:float=30,
                             ):
        '''
        Функция использует:
            данные self.data_U содержащие сведения о длинах волн максимумов из файла с параметрами U
            self.spectra.maxima_values - значения длин волн максимумов из спектра поглащения
            self.shift - сдвиг между положениями максимумов в data_U и max_from_spectra 
            (полученные из self.define_shift(tol=tol))
            del_tol - порог для удаления максимума из рассмотрения
                (проверяется есть ли максимумы слева и справа в диапазоне от +-del_tol от выделенного максимума)
            merge_tol - порог объединения максимов поглащения в группы, работает аналогично del_tol.
                

        Функция возвращает DataFrame хранящий только значения пригодные для рассмотрения:
            т.е. он удаляет все данные связанные с переходами не представленными одновременно
            и в self.data_U и спектре поглощения (в данных self.spectra.maxima_values).
            
            Те максимумы, которые имеют одну и туже длинну волны или находятся внутри диапазона +-merge_tol,
            помечаются как отдельная группа.
            ТУТ ЕСТЬ О ЧЕМ ЕЩЕ ПОДУМАТЬ...

        Работа del_tol: 
            Если к некоторому максимуму из self.data_U (с учетом необходимого сдвига self.shift) нет ни одного максимума
            из self.spectra.maxima_values внутри диапазона +-del_tol, то такой максимум убирается из рассмотрения.

        '''
        new_df = self.U_data.copy()
        corrected_max_from_U = new_df.Lambda.values+self.shift
        preliminary_max = []
        
        for max_from_U in corrected_max_from_U:
            dif = abs(self.spectra.maxima_values-max_from_U)
            if min(dif)>del_tol:
                preliminary_max.append(np.nan)
            else:
                preliminary_max.append(self.spectra.maxima_values[dif.argmin()])

        new_df['Preliminary_spectra_max'] = preliminary_max
        new_df = new_df.dropna()
        
        ############################################################################
        
        pre_max = new_df.Preliminary_spectra_max.values
        counter = 0 
        group_idx = [0]
        for idx in range(1, pre_max.shape[0]):
            flag = abs(pre_max[idx]-pre_max[idx-1])<merge_tol
            if not flag:
                counter+=1
            group_idx.append(counter)
            
        new_df['Group']=group_idx
        
        new_df['Grouped'] = new_df.Group.duplicated(keep=False)

        self.corrected_U_data = new_df.copy()
        
        return new_df
    
    def init_boundaries(self, threshold:float=1e-2)->np.ndarray:
        '''
        Функция использует массив значений функции - spectra.y
        список индексов максимумов функции y - spectra.maxima_indexes
        пороговое значение изменения производных для определения границ анализируемой области - threshold
        и возвращает индексы левой и правой границ для каждого максимума функции y в виде np.ndarray([[left1, right1],[left2, right2]...])

        Граница определяется либо по наличию экстремума [x'_i*x'_i-1<=0], либо при слабом изменении производной
        изменение производной считается слабым в случае если:
            x[i] не соответствует экстремуму и при этом sqrt(x'[i]*x'[i+1])<=threshold

        '''
        
        try:
            max_values = self.corrected_U_data.Preliminary_spectra_max.values
        except AttributeError:
            print('Error: attribute "corrected_U_data" not found, use self.get_correct_maxima_df')
            return None
        
        y = self.spectra.y
        maxima_indexes = Spectra.get_indexes_by_values(x=self.spectra.x, values=max_values)
        start_end_indexes = []
        first_derivative = np.append(y[1:]-y[:-1],0)

        for max_position in maxima_indexes:
            left_side = first_derivative[:max_position][::-1]
            right_side = first_derivative[max_position:]
            boundary = list()

            for idx in range(1,len(left_side)-1):
                product_of_nearest_derivative = left_side[idx]*left_side[idx+1]
                is_extremum = product_of_nearest_derivative<=0
                is_negligible = not is_extremum and np.sqrt(product_of_nearest_derivative)<=threshold
                if is_extremum or is_negligible:
                    boundary.append(max_position-idx-1)
                    break
            
            if not boundary:
                boundary.append(max_position-idx-1)

            for idx in range(1,len(right_side)-1):
                product_of_nearest_derivative = right_side[idx]*right_side[idx+1]
                is_extremum = product_of_nearest_derivative<=0
                is_negligible = not is_extremum and np.sqrt(product_of_nearest_derivative)<=threshold
                if is_extremum or is_negligible:
                    boundary.append(max_position+idx+1)
                    break
            
            if len(boundary)==1:
                boundary.append(max_position+idx+1)

            start_end_indexes.append(boundary)

            
        start_end_indexes = np.array(start_end_indexes)
        groups = self.corrected_U_data.Group.values
        
        for j in set(groups):
            start_end_indexes[np.where(groups==j)] = start_end_indexes[np.where(groups==j)][:,0].min(), start_end_indexes[np.where(groups==j)][:,1].max()
    
    
        self.boundaries = start_end_indexes
        return self.boundaries
    
    def set_boundaries_and_fit(self, merge_range:float=30.)->pd.DataFrame:
        
        '''
        Функция использует аттрибуты corrected_U_data и boundaries,
        для инициализации аттрибута corrected_U_data необходимо задействовать метод get_correct_maxima_df
        для инициализации аттрибута boundaries необходио задействовать метод init_boundaries
        
        Параметр merge_range необходим для финального определения индексов сгруппированных термов:
            если ближайшие lambda_mean сгрупперованных термов находятся ближе merge_range,
            то они объединяются в 1 группу, в противном случае им присваеваются разные номера групп.
        
        Функция позволяет задать корректные границы анализируемых областей на спектре поглащения
        А также указать необходимое количество функций Гаусса для каждой анализируемой области.
        Если установить Num_Gauss=0, то соответствующий сегмент убирается из рассмотрения.
        Чекбокс Paired используется для указания существенного пересечения двух максимумов поглащения
        аффилированных с различными термами, в таком случае необходимо выделить сегмент включающий
        области близких максимумов поглащения и поставить соответствующее значение в Num_Gaus,
        в качестве необходимого набора параметров Гаусса будут браться те, что соответствуют первым n установленным
        руками максимумам.
        
        Чекбокс Grouped используется для указания сгрупперованности термов.
        
        Paired подразумевает что вклады термов в спектр поглощения могут быть условно однозначно разделены в отличии
        от Grouped
        '''
        
        assert hasattr(self,'corrected_U_data'),'Error: attribute "corrected_U_data" not found, use self.get_correct_maxima_df'
        
        new_df = self.corrected_U_data.copy()
        
        try:
            boundaries = self.boundaries
        except:
            print('boundaries was not found')
            print('boundaries will generated with self.init_boundaries(threshold=1e-2)')
            boundaries = self.init_boundaries()

        new_df = new_df.reset_index()
        new_df.drop('index', axis=1, inplace=True)
        new_df['Boundaries_start_idx'] = pd.Series(dtype=np.int32)
        new_df['Boundaries_end_idx'] = pd.Series(dtype=np.int32)
        new_df['Num_Gauss'] = pd.Series(dtype=np.int32)
        new_df['Paired'] = pd.Series(dtype=bool)

        new_df['Grouped'] = pd.Series(dtype=bool)
        
        new_df['Gauss_parameters'] = pd.Series(dtype=object)
        new_df['Bkg_parameters'] = pd.Series(dtype=object)
        new_df['Lambda_mean'] = pd.Series(dtype=np.float64)
        new_df['Gauss_AUC'] = pd.Series(dtype=np.float64)
        tic = 0

        segment_generator = (Plot_Segment(spectra_args=self.spectra.x,
                             spectra_values = self.spectra.y,
                             max_wls=self.corrected_U_data.Preliminary_spectra_max.iloc[idx],
                             boundaries=boundaries[idx],
                             grouped=self.corrected_U_data.Grouped.iloc[idx],
                             mean_from_U = self.corrected_U_data.Lambda.iloc[idx]+self.shift,
                             text=self.corrected_U_data.Term.iloc[idx]) for idx in range(self.corrected_U_data.shape[0]))

        t = next(iter(segment_generator))

        def onclick(cliced):
            nonlocal t
            nonlocal tic
            nonlocal new_df 
            assert t.accepted, 'Boundaries was not accepted'
            new_df.at[tic,'Boundaries_start_idx']=int(t.results['Boundaries_start_idx'])
            new_df.at[tic,'Boundaries_end_idx']=int(t.results['Boundaries_end_idx'])
            new_df.at[tic,'Num_Gauss']=int(t.results['Num_Gauss'])
            new_df.at[tic,'Paired']=bool(t.results['Paired'])

            new_df.at[tic,'Grouped']=bool(t.results['Grouped'])
            
            new_df.at[tic,'Gauss_parameters'] = t.results['Gauss_parameters']
            new_df.at[tic,'Bkg_parameters'] = t.results['Bkg_parameters']
            new_df.at[tic,'Lambda_mean'] = t.results['Lambda_mean']
            new_df.at[tic,'Gauss_AUC'] = t.results['Gauss_AUC']
            tic+=1
            if tic<=self.corrected_U_data.shape[0]-1:
                t = next(iter(segment_generator))
                display(button_next, t)
            else:
                new_df.Num_Gauss = new_df.Num_Gauss.replace(0,np.nan)
                new_df.dropna(inplace=True)
                new_df.reset_index(inplace=True)
                new_df.drop('index', axis=1, inplace=True)
                
                grouped_array = new_df.Grouped.values
                lm = new_df.Lambda_mean.values
                counter = 0 
                group_idx = [0]
                for idx in range(1, grouped_array.shape[0]):
                    flag = (int(grouped_array[idx])*int(grouped_array[idx-1])==1) and (abs(lm[idx]-lm[idx-1])<merge_range)  ## TUT DODELAT'!
                    if not flag:
                        counter+=1
                    group_idx.append(counter)
                    
                new_df['Group'] = group_idx
                
                clear_output(wait=True)
                
                display(new_df)
                self.result_df = new_df.copy()
                return new_df

        button_next = widgets.Button(description='Next')
        button_next.on_click(onclick)

        display(button_next, t)

    
    @staticmethod
    def calculate_concentration(mass:float, nu_REI:float, rho_media:float, mol_mass:float, nu_ox:float, mass_media:float):
        '''
        mass - масса вводимого в стекло оксида активатора (г)
        nu_REI - кол-во молей введенного в стекло иона активатора (моль)
        rho_media - плотность стекла (г/см3)
        mol_mass - молярная масса вводимого в стекло оксида активатора (г/моль)
        nu_ox - кол-во молей введенного в стекло оксида активатора (моль)
        mass_media - масса стекла (г)
        '''
        N_a = 6.022*1e23 # (моль-1)
        concentration = (mass*nu_REI*rho_media*N_a)/(mol_mass*nu_ox*mass_media)
        return concentration
    
    def set_concentration(self, conc:float):
        '''Инициализирует аттрибут concentration значением conc'''
        self.concentration = conc
        return None

    
    def set_refractive_index_Selmeier_if_df(self,B:np.ndarray,C:np.ndarray):
        '''
        Функция применяет формулу Селмайера для определения показателя преломления
        на длинне волны соответствующей максимумам поглощения в спектре
        и добовляет соответствующий столбец данных в self.result_df
        '''
        
        def Sellmeier_equation(B:np.ndarray,C:np.ndarray,wl:float)->np.ndarray:
            '''
            B и C трехкомпонентные вектора строки, значения которых подгоняются по эксперименту
            wl - длинна волны
            1+Sum_i(B_i*wl**2/(wl**2-C))
            '''
            numerator = B*(wl**2)
            denominator = (wl**2)-C
            summed_ratio = (numerator/denominator).sum()
            n_squared = 1+summed_ratio
            return np.sqrt(n_squared)
        
        self.result_df['n']= self.result_df.Lambda_mean.apply(lambda x: np.round(Sellmeier_equation(B=B, C=C, wl=np.round(x,3)),10))
        return self.result_df
    
    def set_constant_refractive_index(self, n:float=1.5):
        '''
        Функция устанавливает значение показателя преломления
        для всех рассматриваемых термов равным заданному числу n'''
        self.result_df['n'] = n
        return self.result_df

    def set_line_stranges_in_df(self)->pd.DataFrame:
        '''
        Функция использует:
            аттрибут self.concentration экземпляра Judd_Olfed
                для его инициализации используйте self.set_concentration
            аттрибут self.result_df с столбцом данных 'n'
                для его инициализации используйте self.set_refractive_index_Selmeier_if_df
                или self.set_constant_refractive_index
                
        Функция расчитывает силы линии осцилляторов для данных из self.result_df 
        и сохроняет результаты в новом столбце Line_strange
        '''
        
        def get_conversion_coef(J:float,n:float,lambda_mean,rei_conc):
            coef = 10.413487*(2*J+1)*n*9/(lambda_mean*rei_conc*(n**2+2)**2)
            return coef

        assert hasattr(self,'concentration'), 'Error: attribute "concentration" not found, use self.set_concentration'
        assert hasattr(self.result_df,'n'),"result_df haven't column 'n'"        
        
        new_df = self.result_df.copy()
        new_df['Line_strange'] = pd.Series(dtype='float64')
        new_df['Coef'] = pd.Series(dtype='float64')

        for idx in range(len(new_df)):
            coef = get_conversion_coef(J=new_df.J_f.iloc[idx],
                                       n=new_df.n.iloc[idx],
                                       lambda_mean = new_df.Lambda_mean.iloc[idx],
                                       rei_conc=self.concentration)
            
            new_df.at[idx, 'Coef'] = np.round(coef,10)
            if new_df.Grouped.iloc[idx]:
                new_df.at[idx, 'Line_strange'] = np.nan
            else:
                l_s = coef*new_df.Gauss_AUC.iloc[idx]
                new_df.at[idx, 'Line_strange'] = np.round(l_s,10)
            
        self.result_df = new_df.copy()
        
        return self.result_df
    
    @staticmethod
    def rmse_abs(f_exp:np.ndarray, f_calc:np.ndarray,num_oscillator_strengths, num_fitted_params = 3)->float:
        '''
        num_fitted_params for J-O is equal to 3
        num_oscillator_strengths must be greater than 3
        '''

        assert f_exp.shape[0]==f_calc.shape[0]==num_oscillator_strengths, 'Wrong num_oscillator_strengths of functions shapes'
#         assert num_oscillator_strengths>3, 'num_oscillator_strengths must be greater than 3'

        coef = 1 #1/(num_oscillator_strengths-num_fitted_params)
        error = ((f_exp-f_calc)**2).sum()
        return np.sqrt(coef*error)

    @staticmethod
    def rmse_rel(f_exp:np.ndarray, f_calc:np.ndarray,num_oscillator_strengths, num_fitted_params = 3)->float:
        '''
        num_fitted_params for J-O is equal to 3
        num_oscillator_strengths must be greater than 3
        '''

        assert f_exp.shape[0]==f_calc.shape[0]==num_oscillator_strengths, 'Wrong num_oscillator_strengths of functions shapes'
#         assert num_oscillator_strengths>3, 'num_oscillator_strengths must be greater than 3'

        coef = 1 #1/(num_oscillator_strengths-num_fitted_params)
        error = (((f_exp-f_calc)/(f_exp+1e-10))**2).sum()
        return np.sqrt(coef*error)
    
    
    def fit_omegas(self, loss:str='abs', fit_by:str='ls', omegas:np.ndarray=None)->tuple:
        '''
            Функция использует аттрибут self.result_df который уже должен  содержать столбец Line_strange
            Параметр loss может принимать значения "abs" и "rel" 
            которые указывают какую функцию невязки использовать при фиттинге параметров Омега
            Параметр fit_by указывает по каким данным будет осуществляться фиттинг: 'ls'-силы линий, 'AUC'-площади.
            Стоит отметить, что в текущей реализации вклады Grouped термов по факту считаются через AUC.
            Также в функцию можно передать omegas:np.ndarray - стартовые значения для фиттинга
            Функция выполняет подгонку параметров Джадда-Офельта (Омега2, Омега4, Омега6),
            полученные результаты вносятся в result_df, 
            а полученные значения Омега2, Омега4, Омега6 сохраняются в аттрибуте self.omegas.
        '''
        assert hasattr(self.result_df, 'Line_strange'), 'self.result_df havent column "Line_strange", use self.set_line_stranges_in_df'
        assert loss in ('abs','rel'), 'loss must be "abs" or "rel"'
        assert fit_by in ('ls','AUC'), 'fit_by must be "ls" or "AUC"'
        loss_funcs = {'abs':Judd_Ofelt.rmse_abs,'rel':Judd_Ofelt.rmse_rel}

        base_data = self.result_df.query('Grouped==False')
        base_U = base_data[['U2','U4','U6']].values
        base_l_s = base_data.Line_strange.values
        base_coef = base_data.Coef.values
        base_AUC = base_data.Gauss_AUC.values
        base_nos = len(base_data)

        uniques = self.result_df.query('Grouped==True').Group.unique()
        grouped_dfl  = [self.result_df.query(f'Group=={unique}') for unique in uniques]

        omegas = omegas if omegas else np.array([1,1,1])

        func_j_o = lambda U,omegas: U@omegas


        def error(params):
            params = abs(params)
            errors = []
            for df in grouped_dfl:
                t_U = df[['U2','U4','U6']].values
                ls_calc = func_j_o(t_U,params)
                AUC_calc = ls_calc/df.Coef.values
                auc_sum = AUC_calc.sum()
                err = np.sqrt((df.Gauss_AUC.mean()-auc_sum)**2)
                errors.append(err)
                
            if fit_by=='ls':
                base_errors = loss_funcs[loss](f_exp=base_l_s, f_calc=func_j_o(base_U,params), num_oscillator_strengths=base_nos)
            else:
                base_errors = loss_funcs[loss](f_exp=base_AUC, f_calc=func_j_o(base_U,params)/base_coef, num_oscillator_strengths=base_nos)
            return base_errors+sum(errors)

        res = minimize(error, omegas, method='BFGS', options={'gtol': 1e-24, 'disp': True})

        new_omegas = np.round(abs(res.x),10)
        new_df = self.result_df.copy()
        U_matrix = self.result_df[['U2','U4','U6']].values
        new_df['Theor_line_strange'] = U_matrix@new_omegas
        new_df['Theor_AUC'] = new_df['Theor_line_strange']/self.result_df.Coef.values
        
        self.result_df = new_df.copy()
        self.omegas = {'Omega2':new_omegas[0], 'Omega4':new_omegas[1], 'Omega6':new_omegas[2]}
        return self.omegas
    
    @staticmethod
    def get_help():
        print("""
        Для корректной работы Judd_Ofelt'а необходимо инициализировать объекты:
        Data_U и Spectra:
        
        data_U = Data_U(path_to_U_data)  # При необходимости можно вызвать подсказку
        
        spectra = Spectra(path_to_spectra, delimiter=';')
        Для удобства можно отрисовать спектр с помощью
        spectra.plot_spectra()
        и обрезать его с помощью
        spectra.crop_spectra(x_start=320, x_end=1700)
        
        Также после инициализации у спектра необходимо установить значения максимумов
        m_val = spectra.set_maxima(tolerance=3, max_from_U=data_U.data_U.Lambda.values)
        После выполнения set_maxima у экземпляра spectra инициализируются 2 аттрибута:
        spectra.maxima_indecies и spectra.maxima_values необходимые для Judd_Ofelt'а
        
        После указанных процедур можно инициализировать экземпляр Judd_Ofelt'а
        jo = Judd_Ofelt(U_data=data_U, spectra=spectra)
        (Данные хранятся в self.U_data и self.spectra)
        
        Далее необходимо определить сдвиг между положениями максимумов из data_U и spectra:
        jo.define_shift(tol=30.)                      # Подробней см. в подсказке
        (Результат хранится в self.shift)
        
        После чего можно удалить нерелевантные данные из data_U:
        jo.get_correct_maxima_df(del_tol=40, merge_tol=10) # Подробней см. в подсказке
        (Результат хранится в self.corrected_U_data)
        
        Для дальнейшей работы необходимо установить границы анализируемых областей:
        jo.init_boundaries()                 # инициализирует стартовые положения границ для областей
        Далее нужно вызвать метод self.set_boundaries_and_fit, который
        необходим для ручной проверки установленных границ и их исправления,
        а также для определения параметров фиттинга (Число Гауссовых пиков, спаренность максимумов)
        и его осуществления. 
        В процессе фиттинга определяются площади под кривой поглащения на соответствующем терме (AUC)
        и средние положения длин волн соответствующих максимумов.
        ВАЖНО: Для термов объедененных в группу считается суммарная площадь,
        Для каждого терма из группы нужно проводить фиттинг, при дальнейшем анализе
        суммурные площади по группе усредняются.
        jo.set_boundaries_and_fit()                   # Подробней см. в подсказке
        (Результаты хранятся в self.result_df)
        
        Для определения параметров Джадда Офельда необходимо значения сил линий,
        для определения которых требуются сведения о показателе преломления на заданной длине волны,
        а также значение концентрации РЗИ в составе стекла.
        
        Для установки константного показателя преломления используйте:
        jo.set_constant_refractive_index(n=1.5)
        Для использования показателя преломления определенного в соответствии с ур-ем Селмайера:
        jo.set_refractive_index_Selmeier_if_df(B,C)    # Подробней см. в подсказке
        
        Посчитать концентрацию можно с помощью:
        conc =  jo.calculate_concentration(mass, nu_REI, rho_media, mol_mass, nu_ox, mass_media)
        Для установки концентрации:
        jo.set_concentration(conc=conc)
        (Результат хранится в self.concentration)
        
        После указанных процедур можно рассчитать силы линий с помощью:
        jo.set_line_stranges_in_df()                   # Подробней см. в подсказке
        (Результаты работы хранятся в self.result_df)
        
        ВАЖНО! Т.к. аттрибут result_df является pandas.DataFrame,
        то к нему применимы все методы DataFrame'ов, в частности round.
        Т.е. если необходимо округлить все столбцы в result_df до 4 знака,
        то можно использовать: 
        jo.result_df = jo.result_df.round(decimals=4)
        
        Для определения параметров Omega2,4,6:
        jo.fit_omegas(loss='abs')                     # Подробней см. в подсказке
        Расчетные значения для сил линий сохраняются в self.result_df,
        а вычисленные значение Omega хранятся в self.omegas
        
        """)
        return None


In [8]:
class JuddOfeltAnalyzer:
    def __init__(self, 
                 df: pd.DataFrame,
                 ion: str = "",
                 name: str | None = None):

        """
        Класс предназначен для оценки стабильности полученных значений параметров интенсивности J-O.
        Для инициализации необходимо передать DataFrame с результатами расчетов класса Judd_Ofelt (self.result_df)
        """
        
        self.df = df.copy()
        self.ion = ion
        self.name = name or "Unnamed_Sample"
        
        if 'Group' not in self.df.columns:
            raise ValueError("DataFrame должен содержать колонку 'Group'")
            
        self.unique_groups = np.sort(self.df['Group'].unique())
        
        # Поля для хранения результатов
        self.bootstrap_results: pd.DataFrame | None = None
        self.loo_results: pd.DataFrame | None = None
        self.last_fitted_omegas: np.ndarray | None = None

    def estimate_noise(self, reference_df: pd.DataFrame) -> np.ndarray:
        """Оценка шума по Gauss_AUC."""
        return self._compute_group_auc_differences(self.df, reference_df)

    @staticmethod
    def _compute_group_auc_differences(df1: pd.DataFrame, df2: pd.DataFrame) -> np.ndarray:
        diffs = []
        for g in np.sort(df1['Group'].unique()):
            auc1 = df1.query(f'Group == {g}')['Gauss_AUC'].values
            auc2 = df2.query(f'Group == {g}')['Gauss_AUC'].values
            rms = np.sqrt(((auc1 - auc2) ** 2).mean())
            diffs.append(rms)
        return np.array(diffs)

    def _get_values_by_group(self, omegas: np.ndarray, target: str = 'AUC'):
        exp_values = []
        theor_values = []
        for g in self.unique_groups:
            temp = self.df.query(f'Group == {g}')
            U = temp[['U2', 'U4', 'U6']].values
            ls_calc = U @ omegas
            
            if target == 'AUC':
                theor_val = (ls_calc / temp['Coef'].values).sum()
                exp_val = temp['Gauss_AUC'].mean()
            else:
                theor_val = ls_calc.sum()
                exp_val = (temp['Gauss_AUC'] * temp['Coef']).mean()
            
            exp_values.append(exp_val)
            theor_values.append(theor_val)
        return np.array(exp_values), np.array(theor_values)

    def _loss_function(self, params, target, loss, fixed_perturbation=None):
        params = np.abs(params)
        exp_val, theor_val = self._get_values_by_group(params, target)
        
        if fixed_perturbation is not None:
            exp_val = exp_val + fixed_perturbation
        
        if loss == 'abs':
            error = np.sum((exp_val - theor_val) ** 2)
        else:
            error = np.sum(((exp_val - theor_val) / (exp_val + 1e-12)) ** 2)
        return np.sqrt(error)

    def fit(self,
            initial_omegas: np.ndarray,
            target: Literal['AUC', 'LS'] = 'AUC',
            loss: Literal['abs', 'rel'] = 'abs',
            noise: np.ndarray | float | None = None,
            noise_scale: float = 1.0,
            rng: np.random.RandomState | None = None,
            **minimize_kwargs) -> np.ndarray:
        
        if initial_omegas is None:
            raise ValueError("initial_omegas обязателен.")

        if isinstance(noise, (int, float)):
            noise = np.array([
                self.df.query(f'Group == {g}')['Gauss_AUC'].mean() * noise 
                for g in self.unique_groups
            ])

        # Фиксируем шум на время оптимизации
        fixed_perturbation = None
        if noise is not None:
            if rng is None:
                rng = np.random.RandomState()
            fixed_perturbation = rng.normal(0, noise_scale * noise)

        def objective(params):
            return self._loss_function(params, target, loss, fixed_perturbation)

        res = minimize(
            objective,
            initial_omegas,
            method='BFGS',
            options={'gtol': 1e-12, 'disp': False, **minimize_kwargs}
        )
        
        return np.abs(np.round(res.x, decimals=10))

    def bootstrap(self,
                  n_samples: int = 500,
                  initial_omegas: np.ndarray = None,
                  target: Literal['AUC', 'LS'] = 'AUC',
                  loss: Literal['abs', 'rel'] = 'abs',
                  noise: np.ndarray | float | None = None,
                  noise_scale: float = 1.0,
                  random_state: int | None = None) -> pd.DataFrame:
        """
        Bootstrap-оценка параметров Ω при случайном зашумлении спектра (изменения площадей под полосами поглощения).
        """
        if initial_omegas is None:
            raise ValueError("initial_omegas обязателен для bootstrap.")
        
        rng = np.random.RandomState(random_state) if random_state is not None else np.random.RandomState()
        
        results = {'Omega2': [], 'Omega4': [], 'Omega6': []}
        
        for _ in tqdm(range(n_samples), desc=f"Bootstrap {self.name}"):
            omegas = self.fit(
                initial_omegas=initial_omegas,
                target=target,
                loss=loss,
                noise=noise,
                noise_scale=noise_scale,
                rng=rng
            )
            results['Omega2'].append(omegas[0])
            results['Omega4'].append(omegas[1])
            results['Omega6'].append(omegas[2])
            
        return pd.DataFrame(results)

    def loo_bootstrap(self,
                      n_samples: int = 100,
                      initial_omegas: np.ndarray = None,
                      target: Literal['AUC', 'LS'] = 'AUC',
                      loss: Literal['abs', 'rel'] = 'abs',
                      noise: np.ndarray | float | None = None,
                      noise_scale: float = 1.0,
                      random_state: int | None = None) -> pd.DataFrame:
        """
        Leave-One-Out Bootstrap параметров Ω при случайном зашумлении спектра (изменения площадей под полосами поглощения).
        В процессе работы поочередно убирается одна "Группа" или Полоса поглощения из спектра, а оставшиеся линии зашумляются.
        """
        if initial_omegas is None:
            raise ValueError("initial_omegas обязателен.")
        
        rng = np.random.RandomState(random_state) if random_state is not None else np.random.RandomState()
        
        results = {'Omega2': [], 'Omega4': [], 'Omega6': []}
        
        outer_pbar = tqdm(self.unique_groups, desc=f"LOO Bootstrap {self.name}", leave=True)
        
        for g_out in outer_pbar:
            tdf = self.df[self.df['Group'] != g_out].copy()
            temp_analyzer = JuddOfeltAnalyzer(tdf, self.ion, f"{self.name}_wo_g{g_out}")
            
            if isinstance(noise, np.ndarray):
                current_noise = np.delete(noise, np.where(self.unique_groups == g_out)[0][0])
            else:
                current_noise = noise

            inner_pbar = tqdm(range(n_samples), desc=f"  Group {g_out} excluded", leave=False)
            
            for _ in inner_pbar:
                omegas = temp_analyzer.fit(
                    initial_omegas=initial_omegas,
                    target=target,
                    loss=loss,
                    noise=current_noise,
                    noise_scale=noise_scale,
                    rng=rng
                )
                results['Omega2'].append(omegas[0])
                results['Omega4'].append(omegas[1])
                results['Omega6'].append(omegas[2])
                
        self.loo_results = pd.DataFrame(results)
        return self.loo_results

    
    def get_results(self, 
                    omegas: np.ndarray,
                    target: Literal['AUC', 'LS'] = 'AUC') -> tuple[pd.DataFrame, dict]:
        """
        Возвращает детальную таблицу краткое результаты для данных omegas.
        """
        exp, theor = self._get_values_by_group(omegas, target)
        
        detailed = pd.DataFrame({
            'Group': self.unique_groups,
            'Exp': np.round(exp, 4),
            'Calc': np.round(theor, 4),
            'Diff': np.round(exp - theor, 4),
            'Rel_Diff_%': np.round(100 * (exp - theor) / (exp + 1e-12), 2)
        })
        
        rmse = self._loss_function(omegas, target, 'abs')
        
        summary = {
            'name': self.name,
            'ion': self.ion,
            'omegas': omegas.tolist(),
            'rmse': round(rmse, 5),
            'n_groups': len(self.unique_groups),
            'target': target
        }
        
        self.last_fitted_omegas = omegas
        return detailed, summary
        
    def plot_omega_distributions(self,
                                 bootstrap_df: pd.DataFrame,
                                 reference: dict[str, float]):
        """Визуализация распределений параметров Ω."""
        long_df = pd.melt(
            bootstrap_df,
            value_vars=['Omega2', 'Omega4', 'Omega6'],
            var_name='Omega',
            value_name='Value'
        )

        omega_names = ['Omega2', 'Omega4', 'Omega6']
        fig, axes = plt.subplots(1, len(omega_names), figsize=(5 * len(omega_names), 6))

        for i, omega_name in enumerate(omega_names):
            ax = axes[i] if len(omega_names) > 1 else axes # Handle single subplot case
            data = long_df[long_df['Omega'] == omega_name]
            sns.violinplot(data=data, x='Omega', y='Value', ax=ax)

            ref_val = reference[omega_name]
            ax.axhspan(ref_val * 0.9, ref_val * 1.1, alpha=0.2, color='gray', label='±10%')
            ax.axhline(ref_val, color='red', linestyle='--', label=f'Ref = {ref_val:.4f}')

            ax.set_title(f'{self.name} — {omega_name}')
            ax.set_xlabel('Omega')
            ax.set_ylabel('Value')
            ax.legend()

        plt.tight_layout()
        plt.show()

In [9]:
class RadiativeTransitionCalculator:
    """
    Конвейер для расчёта радиационных характеристик редкоземельных ионов
    по теории Джадда-Офельта (Judd-Ofelt).
    """

    def __init__(self, transitions_df: pd.DataFrame, refractive_index: float = 1.55):
        """
        Args:
            transitions_df: DataFrame с колонками:
                - TERM_i: обозначение перехода, например '4F3/2'
                - U2, U4, U6: матричные элементы
                - A_md_pre: предрассчитанный магнитный дипольный вклад (опционально)
                - Lambda: длина волны в нм (опционально, будет рассчитана)
                - J_i: квантовое число J верхнего уровня
            refractive_index: показатель преломления среды
        """
        self.df = transitions_df.copy()
        self.n = refractive_index

        self._validate_input()

    def _validate_input(self):
        required = ['TERM_i', 'U2', 'U4', 'U6','Lambda','J_i']
        missing = [col for col in required if col not in self.df.columns]
        if missing:
            raise ValueError(f"Отсутствуют обязательные колонки: {missing}")

    def _S_calc(self, omegas: List[float]) -> np.ndarray:
        """Расчёт силы линии S_ed = U·Ω"""
        U_matrix = self.df[['U2', 'U4', 'U6']].values
        return U_matrix @ np.array(omegas)

    @staticmethod
    def _local_field_correction(n: float) -> float:
        """Локальное поле коррекции для электрического дипольного перехода"""
        return n * ((n**2 + 2) / 3)**2

    def _calc_A_ed(self, S_calc: np.ndarray, lambda_nm: np.ndarray, J_upper: np.ndarray) -> np.ndarray:
        """Электрический дипольный коэффициент Эйнштейна A_ed"""
        correction = self._local_field_correction(self.n)
        lambda_cm = lambda_nm * 1e-7  # в см
        jlam = (2 * J_upper + 1) * lambda_cm**3
        
        coef = 7.23e10
        return coef * correction / jlam * (S_calc * 1e-20)

    def _calc_A_md(self) -> np.ndarray:
        """Магнитный дипольный вклад"""
        if 'A_md_pre' not in self.df.columns:
            return np.zeros(len(self.df))
        return self.n**3 * self.df['A_md_pre'].values

    def calculate(self, 
                  omegas: Union[List[float], Tuple[float, float, float]], 
                  sample_name: Optional[str] = None) -> pd.DataFrame:
        """
        Основной метод расчёта.
        
        Returns:
            DataFrame с колонками: Term, Ajj, Bjj, Lifetime_ms, Lambda, ...
        """
        
        self.df['S_calc'] = self._S_calc(omegas)
        self.df['n'] = self.n
        
        lambda_nm = self.df.get('Lambda')
        
        A_ed = self._calc_A_ed(
            S_calc=self.df['S_calc'].values,
            lambda_nm=lambda_nm.values,
            J_upper=self.df['J_i'].values
        )
        
        A_md = self._calc_A_md()
        
        self.df['A_ed'] = A_ed
        self.df['A_md'] = A_md
        self.df['Ajj'] = A_ed + A_md
        
        # Расчёт времени жизни верхних уровней
        lifetime_df = self._calc_lifetimes()
        
        # Расчёт коэффициентов ветвления
        self.df = self.df.merge(lifetime_df, on='TERM_i', how='left')
        self.df['Bjj'] = self.df['Ajj'] * self.df['Lifetime_s']
        
        result = self.df.copy()
        
        # Финальная таблица
        final_cols = ['Term', 'Ajj', 'Bjj', 'Lifetime_ms', 'Lambda', 'S_calc']
        
        output = result[final_cols].copy()
        output.rename(columns={'Lambda': 'Lambda_nm'}, inplace=True)
        
        if sample_name:
            output['Sample'] = sample_name
            
        return output

    def _calc_lifetimes(self) -> pd.DataFrame:
        """Расчёт радиационного времени жизни для каждого верхнего терма"""
        lifetime_dict = []
        for upper, group in self.df.groupby('TERM_i'):
            total_A = group['Ajj'].sum()
            tau_s = 1.0 / total_A if total_A > 0 else np.nan
            lifetime_dict.append({
                'TERM_i': upper,
                'Lifetime_s': tau_s,
                'Lifetime_ms': tau_s * 1000
            })
        
        return pd.DataFrame(lifetime_dict)
    
    @staticmethod
    def get_filtred_data(terms:List[str], result_df: pd.DataFrame,):
        """Возвращает данные для выбранного списка термов, например ['4F3/2-4I9/2', '4F3/2-4I11/2', '4F3/2-4I13/2'] """
        template_df = pd.DataFrame({'Term':terms})
        t_df = template_df.merge(right=result_df, how='inner', on='Term',)
        return t_df[['Term','Ajj','Bjj','Lifetime_ms','Lambda_nm']]